In [117]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, timezone;


df_telemetry = pd.read_csv("../data/raw/sensors_telemetry.csv")

In [109]:
df_telemetry.head(10)

,sensor_id,installation_id,timestamp,solar_output_w,battery_level_pct,consumption_w,alert_code,region
0,CAM-4678-S1,4678,2025-01-06T21:45:05,NaN,32.8,228.68,FAULT_02,ouest
1,CAM-1187-S1,1187,2025-06-18T05:49:56,NaN,46.4,67.97,OVERCURRENT,adamaoua
2,CAM-4076-S1,4076,12-01-2025 23:34:09,NaN,60.0,26.87,OVERCURRENT,ADAMAOUA
3,CAM-3602-S1,3602,1738984414,NaN,40.3,43.43,ovr_v,Centre
4,CAM-4490-S1,4490,1737526408,0.00,48.8,265.51,FAULT_02,LITTORAL
5,CAM-4574-S1,4574,2025-02-18 09:38:21,1295.44,74.8,132.19,ERR,OUE
6,CAM-2447-S1,2447,1741453110,1050.58,67.9,177.02,LOW_BAT,littoral
7,CAM-2129-S1,2129,1746257202,332.90,49.2,221.23,low_bat,Lit.
8,CAM-1813-S1,1813,2025-06-16 19:23:00,0.00,46.0,506.96,NaN,Ouest
9,CAM-4152-S1,4152,2025-02-01 14:17:37,2520.74,97.1,92.82,NO_SIG,Extrême-Nord


In [106]:
df_telemetry.info()

<class 'pandas.DataFrame'>
RangeIndex: 120960 entries, 0 to 120959
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   sensor_id          120960 non-null  str    
 1   installation_id    120960 non-null  int64  
 2   timestamp          120960 non-null  str    
 3   solar_output_w     68332 non-null   float64
 4   battery_level_pct  120960 non-null  float64
 5   consumption_w      116187 non-null  float64
 6   alert_code         80453 non-null   str    
 7   region             120960 non-null  str    
dtypes: float64(3), int64(1), str(4)
memory usage: 7.4 MB


In [119]:
def parse_timestamp(raw) -> datetime:
    """
    Convertit 3 formats de timestamps en datetime UTC :
    - Unix epoch  : 1738984414
    - ISO 8601    : 2024-01-15T14:30:00
    - DD/MM/YYYY  : 15/01/2024 14:30
    - MM/DD/YYYY
    """
    if pd.isna(raw):
        return datetime.now(timezone.utc)

    raw_str = str(raw).strip()

    # Format Unix epoch (10 chiffres)
    if re.match(r'^\d{10}$', raw_str):
        return datetime.fromtimestamp(int(raw_str), tz=timezone.utc)

    # Format ISO 8601
    for fmt in ("%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%SZ"):
        try:
            return datetime.strptime(raw_str, fmt).replace(tzinfo=timezone.utc)
        except ValueError:
            continue

    # Format DD/MM/YYYY HH:MM
    for fmt in ("%d/%m/%Y %H:%M", "%d/%m/%Y %H:%M:%S"):
        try:
            return datetime.strptime(raw_str, fmt).replace(tzinfo=timezone.utc)
        except ValueError:
            continue
        
    # Format 27-04-2025 09:17:13    
    for fmt in ("%d-%m-%Y %H:%M:%S", "%m-%d-%Y %H:%M:%s"):
        try:
            return datetime.strptime(raw_str, fmt).replace(tzinfo=timezone.utc)
        except ValueError:
            continue     

    # Fallback : maintenant
    print(f"  ⚠️  Timestamp non reconnu : '{raw_str}' → remplacé par now()")
    return datetime.now(timezone.utc)



def normalisation_region(raw:str) -> str:
    if not raw or pd.isna(raw):
        return 'Inconnu'
    
    raw = str(raw).strip().lower()
    raw = raw.title()
    
    if raw in {'Ouest', 'Oue', 'Ouest-Cm'}:
        return 'Ouest'
    if raw in {'Lit.', 'Ltl'}:
        return 'Littoral'
    if raw in {'Nord_Ouest', 'No'}:
        return 'Nord-Ouest'
    if raw in {'E', 'Est-Cm','Est'}:
        return 'Est'
    if raw in {'Ada', 'Adamawa'}:
        return 'Adamaoua'
    if raw in {'Ctr'}:
        return 'Centre'
    if raw in {'Ext-Nord', 'Extreme-Nord', 'En'}:
        return 'Extrême-Nord'
     
    return raw      
    
    
def normalisation_alert(raw:str)->str:
    if not raw or pd.isna(raw) :
        return "AUCUNE"  
    raw = str(raw).strip().upper()
    return raw


df_telemetry['battery_level_pct'] = df_telemetry['battery_level_pct'].clip(0,100)

df_telemetry["solar_output_w"]    = pd.to_numeric(df_telemetry["solar_output_w"], errors="coerce").fillna(0.0)
df_telemetry["battery_level_pct"] = pd.to_numeric(df_telemetry["battery_level_pct"], errors="coerce").fillna(0.0)
df_telemetry["consumption_w"]     = pd.to_numeric(df_telemetry["consumption_w"], errors="coerce").fillna(0.0)

df_telemetry['alert_code'] = df_telemetry['alert_code'].apply(normalisation_alert)    

df_telemetry['region'] = df_telemetry['region'].map(normalisation_region) 


df_telemetry['timestamp'] = df_telemetry['timestamp'].apply(parse_timestamp)

In [ ]:
display(df_telemetry)

,sensor_id,installation_id,timestamp,solar_output_w,battery_level_pct,consumption_w,alert_code,region
0,CAM-4678-S1,4678,2025-01-06 21:45:05+00:00,0.00,32.8,228.68,FAULT_02,Ouest
1,CAM-1187-S1,1187,2025-06-18 05:49:56+00:00,0.00,46.4,67.97,OVERCURRENT,Adamaoua
2,CAM-4076-S1,4076,2025-01-12 23:34:09+00:00,0.00,60.0,26.87,OVERCURRENT,Adamaoua
3,CAM-3602-S1,3602,2025-02-08 03:13:34+00:00,0.00,40.3,43.43,OVR_V,Centre
4,CAM-4490-S1,4490,2025-01-22 06:13:28+00:00,0.00,48.8,265.51,FAULT_02,Littoral
...,...,...,...,...,...,...,...,...
120955,CAM-0048-S1,48,2025-01-02 08:25:29+00:00,1053.39,67.6,245.26,FAULT_01,Centre
120956,CAM-2234-S1,2234,2025-01-27 16:06:27+00:00,1454.19,84.8,77.74,OVR_V,Littoral
120957,CAM-0492-S1,492,2025-03-01 05:31:00+00:00,0.00,47.3,35.58,FAULT_01,Centre
120958,CAM-3814-S1,3814,2025-01-02 06:34:46+00:00,0.01,48.8,289.25,NO_SIG,Nord-Ouest


In [122]:

chemin = "../data/processed/sensors_telemetry.csv"

# La commande d'export
df_telemetry.to_csv(chemin, index=False, encoding='utf-8')

print(f"✅ Fichier sauvegardé avec succès sous : {chemin}")
print(f"📊 Nombre de lignes exportées : {len(df_telemetry)}")

✅ Fichier sauvegardé avec succès sous : ../data/processed/sensors_telemetry.csv
📊 Nombre de lignes exportées : 120960
